In [9]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def read_xyz(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    num_atoms = int(lines[0].strip())
    atom_lines = lines[2:2 + num_atoms]

    symbols = []
    positions = []

    for line in atom_lines:
        parts = line.split()
        symbols.append(parts[0])
        positions.append([float(parts[1]), float(parts[2]), float(parts[3])])

    return symbols, np.array(positions)

def build_molecule(symbols, positions):
    from rdkit.Chem import rdchem
    mol = Chem.RWMol()
    conf = Chem.Conformer(len(symbols))

    for i, (symbol, pos) in enumerate(zip(symbols, positions)):
        atom = Chem.Atom(symbol)
        mol_idx = mol.AddAtom(atom)
        conf.SetAtomPosition(mol_idx, pos)

    mol.AddConformer(conf)

    # Add bonds based on distance
    pt = Chem.GetPeriodicTable()
    for i in range(len(symbols)):
        for j in range(i + 1, len(symbols)):
            dist = np.linalg.norm(positions[i] - positions[j])
            # Use covalent radii heuristic
            ri = pt.GetRcovalent(pt.GetAtomicNumber(symbols[i]))
            rj = pt.GetRcovalent(pt.GetAtomicNumber(symbols[j]))
            if dist < 1.3 * (ri + rj):
                mol.AddBond(i, j, Chem.rdchem.BondType.SINGLE)

    mol = mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol

def get_charge_vector(mol):
    # Add hydrogens before computing charges
    mol_with_H = mol
    AllChem.ComputeGasteigerCharges(mol_with_H)

    charges = []
    for atom in mol_with_H.GetAtoms():
        charge = float(atom.GetProp('_GasteigerCharge'))
        charges.append(charge)

    return np.array(charges)

# Example usage
file_path = 'structures_train/mol_2.xyz'
symbols, positions = read_xyz(file_path)
mol = build_molecule(symbols, positions)
charge_vector = get_charge_vector(mol)
#pot_charge_vec = positions*np.repeat(charge_vector[:, np.newaxis], 3, axis=1)



print("Charge vector:", charge_vector.shape)


Charge vector: (16,)
